In [1]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import umap # <-- UMAP 库
from sklearn.decomposition import PCA
import time

# 设置 matplotlib 在 notebook 中内联显示
%matplotlib inline

In [2]:
# --- 1. 定义文件名 ---
NPZ_FILE = 'training_dataset.npz'
X_SCALER_FILE = 'x_scaler.pkl'
Y_SCALER_FILE = 'y_scaler.pkl'

print(f"正在加载数据从: {NPZ_FILE}")
try:
    data = np.load(NPZ_FILE)
    X = data['X']
    Y = data['Y']
    print(f"原始 X 形状: {X.shape}, 原始 Y 形状: {Y.shape}")
except Exception as e:
    print(f"错误: 无法加载 {NPZ_FILE}。{e}")

print(f"正在加载 Scalers: {X_SCALER_FILE}, {Y_SCALER_FILE}")
try:
    with open(X_SCALER_FILE, 'rb') as f:
        x_scaler = pickle.load(f)
    with open(Y_SCALER_FILE, 'rb') as f:
        y_scaler = pickle.load(f)
    print("Scalers 加载成功。")
except Exception as e:
    print(f"错误: 无法加载 .pkl 文件。请确保它们存在。{e}")

正在加载数据从: training_dataset.npz
原始 X 形状: (8, 12001, 3), 原始 Y 形状: (8, 7)
正在加载 Scalers: x_scaler.pkl, y_scaler.pkl
Scalers 加载成功。


In [3]:
print("正在预处理数据 (压平并归一化)...")

try:
    # 1. 压平 X
    num_samples = X.shape[0]
    # (N, 12001, 3) -> (N, 36003)
    X_flat = X.reshape(num_samples, -1)
    print(f"X 压平后的形状: {X_flat.shape}")

    # 2. 归一化 X 和 Y
    X_scaled = x_scaler.transform(X_flat)
    Y_scaled = y_scaler.transform(Y)
    print("X 和 Y 归一化完成。")

except ValueError as e:
    print(f"错误: Scaler 维度不匹配。")
    print(f"Scaler 期望 {x_scaler.n_features_in_} 个特征, 但数据有 {X_flat.shape[1]} 个。")
    print("请确保您使用的 .pkl 文件是使用当前 NPZ 数据训练的。")
except Exception as e:
    print(f"预处理 X 时出错: {e}")

正在预处理数据 (压平并归一化)...
X 压平后的形状: (8, 36003)
X 和 Y 归一化完成。


In [4]:
print("开始对 X (36003维) 进行降维...")
start_time = time.time()

# 确保样本数足够进行 PCA
n_components_pca = min(50, num_samples - 1)

if n_components_pca > 1:
    print(f"步骤 1: PCA (36003D -> {n_components_pca}D)...")
    pca = PCA(n_components=n_components_pca, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    print(f"PCA 完成 (耗时: {time.time() - start_time:.2f} 秒)")

    print(f"步骤 2: UMAP ({n_components_pca}D -> 2D)...")
    umap_start_time = time.time()
    reducer_x = umap.UMAP(
        n_components=2,     # 降到 2D
        random_state=42,    # 确保结果可复现
        n_neighbors=15,     # 默认值，控制局部/全局结构的平衡
        min_dist=0.1        # 控制点的密集程度
    )
    embedding_x = reducer_x.fit_transform(X_pca) # 在 PCA 结果上运行 UMAP
    print(f"UMAP on X 完成。(耗时: {time.time() - umap_start_time:.2f} 秒)")
else:
    print(f"样本数 ({num_samples}) 太少，无法进行 PCA。跳过 X 的 UMAP。")
    embedding_x = np.random.rand(num_samples, 2) # 创建随机点以便绘图

print(f"X 降维总耗时: {time.time() - start_time:.2f} 秒")

开始对 X (36003维) 进行降维...
步骤 1: PCA (36003D -> 7D)...
PCA 完成 (耗时: 0.02 秒)
步骤 2: UMAP (7D -> 2D)...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


UMAP on X 完成。(耗时: 2.57 秒)
X 降维总耗时: 2.59 秒


In [5]:
print("开始对 Y (7维) 进行降维...")
start_time = time.time()

reducer_y = umap.UMAP(
    n_components=2,
    random_state=42
)
embedding_y = reducer_y.fit_transform(Y_scaled)

print(f"Y 降维完成。(耗时: {time.time() - start_time:.2f} 秒)")

开始对 Y (7维) 进行降维...
Y 降维完成。(耗时: 0.01 秒)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/umap/umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(
